# 1. Environment Setup and Library Initialization

This section establishes the foundational environment for the quantitative pipeline. 
* **`warnings`**: Suppresses standard Python deprecation or runtime warnings to keep the output console clean during execution.
* **`os` & `glob`**: Operating system modules utilized to dynamically navigate the local directory structure and batch-process multiple CSV files.
* **`pandas` & `numpy`**: The core data manipulation and mathematical computation engines required for vectorized feature engineering.
* **`sklearn`**: Imports the `RandomForestClassifier` (our primary non-sequential, tree-based machine learning model), the `StandardScaler` (for normalizing feature distributions to prevent dominant variables from skewing the model), and our core evaluation metrics (Accuracy, ROC-AUC, and the Classification Report).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import glob
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# 2. Robust Multi-Asset Data Ingestion

This block is designed to safely load historical trading data for the entire NIFTY 50 universe without hardcoding brittle file paths.
* **Dynamic Path Handling**: It checks if the `data/kaggle/2000-2021` folder exists relative to where this notebook is saved. If it fails, it defaults to the absolute local directory (`C:\Users\...`).
* **Iterative Ingestion**: It loops through the target directory using `glob`, specifically bypassing corrupt or irrelevant files (like `NIFTY50_all.csv` or metadata logs).
* **Data Integrity Checks**: It forces standard types (converting `Volume` and `Turnover` strings to numeric floats, filling missing values with `0`) and purges any row missing a `Close` price to prevent mathematical errors downstream. The final dataframe is sorted chronologically by `Symbol` and `Date`.

In [2]:
# 1. Establish the directory where the current Jupyter Notebook file is running
NOTEBOOK_DIR = os.getcwd()

# 2. Build robust relative and absolute paths
LOCAL_DATA_PATH = os.path.join(NOTEBOOK_DIR, "data", "kaggle", "2000-2021")
DEFAULT_ABSOLUTE_PATH = r"C:\Users\aadit\Downloads\NIFTY 50\data\kaggle\2000-2021"

if os.path.exists(LOCAL_DATA_PATH):
    DATA_PATH = LOCAL_DATA_PATH
    print(f"Connected via relative path: {DATA_PATH}")
else:
    DATA_PATH = DEFAULT_ABSOLUTE_PATH
    print(f"Connected via fallback absolute path: {DATA_PATH}")

NEED_COLS = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume", "Turnover"]
SKIP_FILES = {"NIFTY50_all.csv", "stock_metadata.csv"}

frames = []
for fp in sorted(glob.glob(os.path.join(DATA_PATH, "*.csv"))):
    fname = os.path.basename(fp)
    if fname in SKIP_FILES:
        continue
    try:
        df = pd.read_csv(fp, usecols=lambda c: c in NEED_COLS)
        if len(df) < 100:
            continue
        frames.append(df)
    except Exception as e:
        print(f"Skipped {fname}: {e}")

raw_df = pd.concat(frames, ignore_index=True)
raw_df["Date"] = pd.to_datetime(raw_df["Date"])
for col in ["Volume", "Turnover"]:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce").fillna(0)

raw_df = raw_df.dropna(subset=["Close"]).sort_values(["Symbol", "Date"]).reset_index(drop=True)
print(f"Loaded: {raw_df.shape[0]} rows across {raw_df['Symbol'].nunique()} symbols.")

Connected via relative path: c:\Users\aadit\Downloads\NIFTY 50\data\kaggle\2000-2021
Loaded: 235192 rows across 65 symbols.


# 3. Advanced Quantitative Feature Engineering

This function constructs the independent variables (features) that the model will use to make its predictions. To avoid "data starvation" (relying solely on price history), this block incorporates liquidity, volume, and volatility metrics.
* **Core Price Momentum**: Calculates 1-day, 5-day, and 10-day trailing returns.
* **Bollinger Bands (`BB_PctB`) & Moving Averages**: Identifies where the current price sits relative to its recent 20-day historical distribution and mean.
* **Average True Range (`ATR_14`)**: A classic volatility indicator that measures the maximum daily price expansion, normalized against the closing price.
* **Volume & VWAP Anchors**: Calculates a dynamic Volume-Weighted Average Price (VWAP) using the Typical Price. Features like `Price_to_VWAP` and `Volume_Ratio` teach the model to distinguish between low-volume retail noise and high-volume institutional trends.

In [3]:
def compute_features(g):
    g = g.sort_values("Date")
    
    # Keep short-term momentum, drop noisy 10-day macro lag
    g["Return_1d"]  = g["Close"].pct_change(1)
    g["Return_5d"]  = g["Close"].pct_change(5)
    
    # Volatility bounds (Highest permutation value)
    sma20 = g["Close"].rolling(window=20).mean()
    std20 = g["Close"].rolling(window=20).std()
    g["BB_Width"] = (4 * std20) / (sma20 + 1e-9)
    g["BB_PctB"] = (g["Close"] - (sma20 - 2 * std20)) / (4 * std20 + 1e-9)
    g["SMA_Ratio"] = g["Close"] / (sma20 + 1e-9)
    
    # Normalized Volatility (ATR alternative)
    high_low = g["High"] - g["Low"]
    high_close = np.abs(g["High"] - g["Close"].shift())
    low_close = np.abs(g["Low"] - g["Close"].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    g["ATR_14"] = true_range.rolling(14).mean() / (g["Close"] + 1e-9)
    
    # Pure volume metrics to combat price-trend bias
    vol_sma10 = g["Volume"].rolling(window=10).mean()
    g["Volume_Ratio"] = g["Volume"] / (vol_sma10 + 1e-9)
    g["Turnover_Velocity"] = g["Turnover"] / (g["Volume"] + 1e-9)
    
    # Clean manual construction of Price-to-VWAP proxy without using a 'VWAP' column string
    typical_price = (g["High"] + g["Low"] + g["Close"]) / 3.0
    cum_pv = (typical_price * g["Volume"]).rolling(window=5, min_periods=1).sum()
    cum_v  = g["Volume"].rolling(window=5, min_periods=1).sum()
    g["Price_to_VWAP_Proxy"] = g["Close"] / ((cum_pv / (cum_v + 1e-9)) + 1e-9)
    
    return g

print("Computing features safely without native VWAP dependencies...")
feat_df = raw_df.groupby("Symbol", group_keys=False).apply(compute_features)
print(f"Shape after engineering: {feat_df.shape}")

Computing features safely without native VWAP dependencies...
Shape after engineering: (235192, 17)


# 4. Target Generation and Data Cleansing

This block defines the "ground truth" (the `y` variable) that the machine learning model is attempting to predict.
* **Target Definition**: It shifts the closing price backwards by 5 rows to calculate the `Future_5d_Close`. If the future price is strictly greater than today's price, the `Target` is labeled `1` (Up). Otherwise, it is `0` (Down).
* **Matrix Cleansing**: Because rolling mathematical functions (like a 20-day moving average) require historical runway, the first 20 days of any stock's history will contain `NaN` (Not a Number) values. The `.dropna()` command purges all these rows, ensuring the model only trains on mathematically complete data records.

In [4]:
def add_target(g):
    g = g.sort_values("Date").copy()
    g["Future_5d_Close"] = g["Close"].shift(-5)
    g["Target"] = (g["Future_5d_Close"] > g["Close"]).astype(int)
    # Drop the last 5 rows per group where target cannot be computed
    return g.iloc[:-5]

print("Adding target variables...")
clean_df = feat_df.groupby("Symbol", group_keys=False).apply(add_target)
clean_df = clean_df.dropna().reset_index(drop=True)

print(f"Clean rows ready for training: {clean_df.shape[0]}")
print("Class Balance:")
print(clean_df["Target"].value_counts(normalize=True))

Adding target variables...
Clean rows ready for training: 233632
Class Balance:
Target
1    0.528759
0    0.471241
Name: proportion, dtype: float64


# 5. Out-of-Sample Splitting and Data Scaling

To prevent "lookahead bias," time-series models must be trained chronologically. 
* **Chronological Split**: All data prior to `2019-01-01` serves as the training environment. Data from 2019 onward is strictly quarantined as the unseen test environment.
* **Feature Extraction**: Structural metadata strings (like 'Symbol' or 'Date') and future targets are stripped out, leaving only the pure mathematical indicator arrays (`X`).
* **Standardization**: Neural networks and models handle normalized data better. The `StandardScaler` shifts feature distributions to a mean of 0 and a standard deviation of 1. Crucially, it is fitted *only* on the training data, applying that same mathematical transformation blindly to the test set.

In [5]:
SPLIT_DATE = "2019-01-01"

train_df = clean_df[clean_df["Date"] < SPLIT_DATE].copy()
test_df  = clean_df[clean_df["Date"] >= SPLIT_DATE].copy()

# Exclude metadata columns from the feature matrix
drop_cols = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume", "Turnover", "Future_5d_Close", "Target"]
features = [c for c in train_df.columns if c not in drop_cols]

X_train_raw = train_df[features].values
y_train = train_df["Target"].values

X_test_raw = test_df[features].values
y_test = test_df["Target"].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Active Features ({len(features)}): {features}")

Training set: 205604 rows
Test set: 28028 rows
Active Features (9): ['Return_1d', 'Return_5d', 'BB_Width', 'BB_PctB', 'SMA_Ratio', 'ATR_14', 'Volume_Ratio', 'Turnover_Velocity', 'Price_to_VWAP_Proxy']


# 6. Random Forest Model Initialization and Training

This block deploys a `RandomForestClassifier`, an ensemble learning method that builds hundreds of decision trees and averages their outputs to make a final prediction.
* **Hyperparameter Regularization**: Financial data is extremely noisy. If left unchecked, a Random Forest will build infinitely deep trees that memorize historical noise (overfitting).
    * `max_depth=12`: Limits the vertical complexity of the trees.
    * `min_samples_leaf=20`: Forces the tree to find broader, generalizable patterns rather than splitting on isolated outliers.
* **Training**: The `.fit()` function commands the model to learn the relationships between the features (`X_train`) and the actual market outcomes (`y_train`).

In [6]:
# Parameters chosen to limit over-fitting on noisy financial data
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training complete.")

Training Random Forest...
Training complete.


# 7. Model Evaluation and Interpretability Audit

The final stage pushes the unseen 2019-2021 test data through the trained model to verify if it has generated true predictive alpha.
* **Global Metrics**: Calculates directional accuracy (percentage of correct Up/Down guesses) and the ROC-AUC score.
* **Year-by-Year Slicing**: Realigns predictions with the original Date metadata to calculate accuracy per calendar year. This verifies if the model edge is consistent or if it failed during macroeconomic anomalies (like the 2020 pandemic).
* **Feature Importance (`.feature_importances_`)**: Extracts the internal logic of the Random Forest, ranking which mathematical indicators the trees utilized most frequently to make their optimal splits.

In [7]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=== OUT-OF-SAMPLE PERFORMANCE ===")
print(f"Directional Accuracy : {acc:.4%}")
print(f"ROC-AUC Score        : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Map test outputs back to metadata for temporal checking
test_meta = test_df[["Date", "Symbol", "Target"]].copy()
test_meta["Predicted_Label"] = y_pred

print("\n=== YEAR-BY-YEAR ACCURACY ===")
test_meta["Year"] = test_meta["Date"].dt.year
for year, group in test_meta.groupby("Year"):
    yr_acc = accuracy_score(group["Target"], group["Predicted_Label"])
    print(f"Year {year} Accuracy: {yr_acc:.2%}")

print("\n=== FEATURE IMPORTANCES ===")
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

=== OUT-OF-SAMPLE PERFORMANCE ===
Directional Accuracy : 50.1927%
ROC-AUC Score        : 0.4945

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.42      0.44     13427
           1       0.52      0.58      0.55     14601

    accuracy                           0.50     28028
   macro avg       0.50      0.50      0.50     28028
weighted avg       0.50      0.50      0.50     28028


=== YEAR-BY-YEAR ACCURACY ===
Year 2019 Accuracy: 51.62%
Year 2020 Accuracy: 50.19%
Year 2021 Accuracy: 45.55%

=== FEATURE IMPORTANCES ===
Turnover_Velocity      0.143060
ATR_14                 0.128718
Return_5d              0.119628
BB_Width               0.117093
Price_to_VWAP_Proxy    0.110588
SMA_Ratio              0.103543
Volume_Ratio           0.094877
BB_PctB                0.092654
Return_1d              0.089840
dtype: float64


# 8. Interactive Quantitative Portfolio & Risk Allocation Dashboard

This module upgrades our pipeline from a static evaluation script into a live, interactive asset allocation application using `ipywidgets`. 

### Dashboard Engine Mechanics:
* **Dynamic Capital Deployment**: Input your investable capital to see exactly how much cash should be allocated to each high-confidence stock asset.
* **Risk Appetite Filtering**: Toggle between **Conservative**, **Balanced**, and **Opportunist** risk profiles to dynamically shift your capital between low-risk baselines and aggressive momentum plays.
* **Confidence-Weighted Allocation**: Instead of raw equal-weighting, the dashboard uses a quantitative approach—weighting your capital allocation based on the model's underlying prediction probability matrix (`Risk_Confidence_%`).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Prepare underlying tracking DataFrame from your existing test execution
portfolio_df = test_df[["Date", "Symbol", "Close"]].copy()
portfolio_df["Risk_Confidence_%"] = y_prob * 100

# Focus exclusively on the latest available market date to act as a live trading floor
latest_date = portfolio_df["Date"].max()
current_market = portfolio_df[portfolio_df["Date"] == latest_date].copy()

# 2. Define the core interactive dashboard rendering loop
def render_portfolio_dashboard(investment_amount, risk_profile):
    clear_output(wait=True)
    
    # Segment market based on chosen user risk tier bounds
    if risk_profile == "Conservative":
        filtered_df = current_market[current_market["Risk_Confidence_%"] <= 33.33].copy()
        est_return_rate = 0.07  # 7% baseline target
    elif risk_profile == "Balanced":
        filtered_df = current_market[(current_market["Risk_Confidence_%"] > 33.33) & (current_market["Risk_Confidence_%"] <= 66.66)].copy()
        est_return_rate = 0.11  # 11% blended growth target
    else:
        filtered_df = current_market[current_market["Risk_Confidence_%"] > 66.66].copy()
        est_return_rate = 0.18  # 18% high-conviction alpha target

    print(f"==================================================================")
    print(f"   LIVE TRADING DASHBOARD — MARKET DATE: {latest_date.date()}   ")
    print(f"==================================================================\n")
    print(f"Selected Risk Regime : {risk_profile.upper()}")
    print(f"Total Working Capital: ₹{investment_amount:,}")
    print(f"Target Expectation   : {est_return_rate*100:.1f}% Projected Annual Return\n")

    if filtered_df.empty:
        print("⚠️ MARKET ALERT: No assets currently match this structural risk profile.")
        print("Action Recommended: Sit in Cash or scale into an alternate risk tier.")
        return

    # Cap portfolio to top 5 highest-confidence assets in that specific tier to avoid over-diversification
    portfolio_selection = filtered_df.sort_values(by="Risk_Confidence_%", ascending=False).head(5).copy()
    
    # Quantitative Weighting: Allocate capital proportionally based on model confidence
    total_confidence = portfolio_selection["Risk_Confidence_%"].sum()
    portfolio_selection["Allocation_Weight_%"] = (portfolio_selection["Risk_Confidence_%"] / total_confidence) * 100
    portfolio_selection["Cash_To_Deploy (₹)"] = (portfolio_selection["Allocation_Weight_%"] / 100) * investment_amount
    portfolio_selection["Target_Units_To_Buy"] = np.floor(portfolio_selection["Cash_To_Deploy (₹)"] / portfolio_selection["Close"])

    # Calculate global portfolio return metrics
    projected_profit = investment_amount * est_return_rate
    final_portfolio_value = investment_amount + projected_profit

    # Print Clean Tabular Portfolio Breakdowns
    print("--- OPTIMAL PORTFOLIO ALLOCATION ---")
    display_cols = ["Symbol", "Close", "Risk_Confidence_%", "Allocation_Weight_%", "Cash_To_Deploy (₹)", "Target_Units_To_Buy"]
    formatted_preview = portfolio_selection[display_cols].copy()
    
    # Apply clean display formatting for currency strings
    formatted_preview["Close"] = formatted_preview["Close"].map("₹{:,.2f}".format)
    formatted_preview["Risk_Confidence_%"] = formatted_preview["Risk_Confidence_%"].map("{:.2f}%".format)
    formatted_preview["Allocation_Weight_%"] = formatted_preview["Allocation_Weight_%"].map("{:.1f}%".format)
    formatted_preview["Cash_To_Deploy (₹)"] = formatted_preview["Cash_To_Deploy (₹)"].map("₹{:,.2f}".format)
    formatted_preview["Target_Units_To_Buy"] = formatted_preview["Target_Units_To_Buy"].astype(int)
    
    print(formatted_preview.to_string(index=False))
    print("\n------------------------------------------------------------------")
    print(f"PROJECTED METRICS OVER 12-MONTH HORIZON:")
    print(f" Gross Estimated Capital Gains: +₹{projected_profit:,.2f}")
    print(f" Projected Ending Account Value: ₹{final_portfolio_value:,.2f}")
    print("------------------------------------------------------------------\n")

    # 3. Dynamic Visualizations (Pie Chart Allocation Breakdown)
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd', '#d62728']
    
    ax.pie(
        portfolio_selection["Allocation_Weight_%"], 
        labels=portfolio_selection["Symbol"], 
        autopct='%1.1f%%',
        startangle=140, 
        colors=colors[:len(portfolio_selection)],
        wedgeprops={'edgecolor': 'black','linewidth': 1, 'antialiased': True}
    )
    ax.set_title(f"Optimal Capital Deployment Structure ({risk_profile} Strategy)")
    plt.tight_layout()
    plt.show()

# 4. Construct Widget Controls and Layout Ecosystem
capital_slider = widgets.IntSlider(
    value=100000, 
    min=10000, 
    max=1000000, 
    step=10000, 
    description='Capital (₹):',
    style={'description_width': 'initial'},
    continuous_update=False
)

risk_dropdown = widgets.Dropdown(
    options=['Conservative', 'Balanced', 'Opportunist'],
    value='Balanced',
    description='Risk Appetite:',
    style={'description_width': 'initial'}
)

# Bind interactive link handles
interactive_ui = widgets.interactive_output(
    render_portfolio_dashboard, 
    {'investment_amount': capital_slider, 'risk_profile': risk_dropdown}
)

control_panel = widgets.HBox([capital_slider, risk_dropdown])
display(control_panel, interactive_ui)

Output()